## Optimization Problem Formulation

We solve a simulation-based optimization problem to determine the optimal operating conditions of the PSA process. The goal is to maximize process productivity while satisfying constraints on product purity, recovery, and cyclic steady-state (CSS) convergence.

The decision variables are defined as

$$
x =
(L,\; P_0,\; \dot{n},\; t_{ads},\; \alpha,\; \beta,\; P_I,\; P_l)
$$

where

| Variable | Meaning |
|--------|--------|
| L | column length |
| P0 | adsorption pressure |
| ndot | feed flow rate |
| tads | adsorption time |
| alpha | pressurization fraction |
| beta | purge fraction |
| PI | intermediate pressure |
| Pl | low pressure |

Each variable is bounded within physically meaningful limits.

---

## Objective Function

The primary objective is to maximize the PSA productivity while optionally penalizing energy consumption.

$$
\max_x f(x)
$$

where

$$
f(x) =
\text{Productivity}(x) -
w_E \cdot \text{Energy}(x)
$$

In the current experiments the energy penalty weight $w_E$ is set to zero, so the optimization effectively maximizes productivity.

---

## Process Constraints

The optimization is subject to several process constraints.

### Purity constraint

$$
\text{Purity}(x) \ge P_{min}
$$

which is implemented as

$$
g_1(x) =
P_{min} - \text{Purity}(x) \le 0
$$

---

### Recovery constraint

$$
\text{Recovery}(x) \ge R_{min}
$$

implemented as

$$
g_2(x) =
R_{min} - \text{Recovery}(x) \le 0
$$

---

### Cyclic steady state (CSS)

The PSA cycle must reach cyclic steady state.

$$
g_3(x) =
CSS(x) - \epsilon_{css} \le 0
$$

where $CSS(x)$ measures the difference between consecutive cycles.

---

### Pressure ordering constraints

To ensure physically meaningful operation, the pressure levels must satisfy

$$
P_I \le P_0
$$

$$
P_l \le P_I
$$

---

## Optimization Method

Because the PSA model is evaluated through a cycle simulation, analytical gradients are not available. Therefore, we use a **trust-region sequential linear programming (SLP)** approach.

At each iteration:

1. The PSA cycle simulator evaluates the current operating point.
2. Gradients of the objective and constraints are estimated using finite differences.
3. A linearized trust-region subproblem is constructed.
4. The subproblem is solved using a linear programming solver.
5. The trust-region radius is updated depending on the improvement of the solution.

The LP subproblem is solved using the **GLPK solver** through Pyomo.

---

## Optimization Algorithm Workflow

The optimization procedure follows a trust-region sequential linear programming (SLP) framework applied to a simulation-based PSA process model.

The overall workflow of the algorithm is illustrated below.


Initial guess $x_0$  
↓  
PSA cycle simulation  
↓  
Evaluate objective and constraints  
↓  
Finite difference gradient estimation  
↓  
Construct trust-region LP subproblem  
↓  
Solve LP (Pyomo + GLPK)  
↓  
Candidate solution $x_{k+1}$  
↓  
Accept / Reject step  
↓  
Update trust region  
↓  
Check convergence


### Algorithm Summary

At each iteration $k$, the algorithm performs the following steps:

1. **Cycle simulation**  
   The PSA process is simulated using the current decision vector $x_k$.

2. **Objective and constraint evaluation**  
   Productivity, purity, recovery, and CSS error are computed from the simulation results.

3. **Gradient estimation**  
   Gradients of the objective and constraint functions are approximated using finite differences.

4. **Trust-region subproblem**  
   A local linear approximation of the nonlinear problem is constructed within a trust region.

5. **Linear programming solve**  
   The linearized subproblem is solved using Pyomo with the GLPK solver.

6. **Trust-region update**  
   The candidate solution is accepted if it improves the objective or reduces constraint violation; otherwise the trust region is reduced.

7. **Iteration**  
   The procedure repeats until convergence criteria are satisfied.

This approach allows efficient optimization of the PSA process even when analytical derivatives of the simulator are unavailable.

In [ ]:
import sys
sys.path.append("..")

from pathlib import Path
import random

from psa_pyomo.model.column_model import ColumnDecisionSpace, VARIABLE_ORDER
from psa_pyomo.process.cycle_model import CycleConfig, CycleSimulator
from psa_pyomo.optimization.optimize_psa import OptimizationConfig, objective_value, run_optimization
from psa_pyomo.run import SCALE
import os
os.environ["PATH"] = f"{os.path.expanduser('~/.juliaup/bin')}:{os.environ['PATH']}"

## Parameters

In [ ]:
solver    = 'glpk'
mat_index = 16
N         = 10      # spatial grid points for PSA simulation
N_START   = 1       # number of random restarts

purity_min   = 0.25
recovery_min = 0.50
css_tol      = 1e-4

max_iter             = 20
delta0               = 0.25
fd_rel_step          = 1e-3
energy_weight        = 0.0
infeasibility_penalty = 1e6

## Decision Variable Bounds

In [ ]:
# Physical bounds (Pa, mol/s, s, dimensionless)
# References:
#   P0  : Haghpanah et al. (2013), Ind. Eng. Chem. Res. — 1.2–3.66 atm for post-combustion CO2 PSA
#   Pl  : Zhang et al. (2008), Adsorption — optimal Pl = 0.025–0.1 bar for zeolite 13X VSA
#   PI  : Kumar (1994), Chem. Eng. Sci.; Nilchan & Pantelides (1998) — ~0.15–0.5 bar intermediate
#   tads: Haghpanah et al. (2013) — optimal tads in 60–200 s range; extended to 400 s
#   alpha, beta: general PSA cycle optimization literature — 0.10–0.40
bounds_physical = {
    'L':    (1.0,   1.0  ),
    'P0':   (1e5,   4e5  ),
    'ndot': (0.5,   2.0  ),
    'tads': (60.0,  400.0),
    'alpha':(0.10,  0.40 ),
    'beta': (0.10,  0.40 ),
    'PI':   (2e4,   2e5  ),
    'Pl':   (2e3,   1e4  ),
}

# Scaled bounds used internally by the optimizer
bounds = {k: (lb / SCALE[k], ub / SCALE[k]) for k, (lb, ub) in bounds_physical.items()}

print("Bounds (physical units):")
for k, (lb, ub) in bounds_physical.items():
    print(f"  {k:>6}: [{lb:.3g}, {ub:.3g}]")

## Initial Point

In [ ]:
# Best purity point from data generation (dataset_material16.csv)
# Sobol quasi-random sampling, 2000 samples, N=10 grid
initial_point_physical = {
    'L':    1.0,
    'P0':   107544.0,
    'ndot': 1.945,
    'tads': 385.6,
    'alpha': 0.190,
    'beta':  0.353,
    'PI':    26011.0,
    'Pl':    7939.0,
}

# Scale for optimizer
initial_point = {k: v / SCALE[k] for k, v in initial_point_physical.items()}

decision_space = ColumnDecisionSpace(bounds=bounds, initial_point=initial_point)
decision_space.validate()
print("Initial point validated.")
print("Starting from best dataset point (purity=0.270)")

In [ ]:
cycle_config = CycleConfig(
    material_index=mat_index,
    n_grid=N,
    css_tol=css_tol,
)

simulator = CycleSimulator(Path('..'), cycle_config)
print(f"Simulator ready (Julia backend, material index={mat_index}, N={N} CSS cycles)")

In [ ]:
optimization_config = OptimizationConfig(
    purity_min=purity_min,
    recovery_min=recovery_min,
    css_tol=css_tol,
    energy_weight=energy_weight,
    fd_rel_step=fd_rel_step,
    delta0=delta0,
    max_iter=max_iter,
    solver_name=solver,
    iter_log_path="../logs/optimization_iterations.csv",
    infeasibility_penalty=infeasibility_penalty,
)

In [ ]:
best_point = None
best_eval  = None

for i in range(N_START):
    print(f"\n=== Start {i+1} / {N_START} ===")

    if i > 0:
        # Random initial point within scaled bounds for restarts
        decision_space.initial_point = {
            k: random.uniform(bounds[k][0], bounds[k][1])
            for k in VARIABLE_ORDER
        }
        # Enforce pressure ordering
        if decision_space.initial_point["PI"] > decision_space.initial_point["P0"]:
            decision_space.initial_point["PI"] = decision_space.initial_point["P0"] * 0.5
        if decision_space.initial_point["Pl"] > decision_space.initial_point["PI"]:
            decision_space.initial_point["Pl"] = decision_space.initial_point["PI"] * 0.5

    try:
        point, evaluation = run_optimization(simulator, decision_space, optimization_config)
        if best_eval is None or evaluation.productivity > best_eval.productivity:
            best_point = point
            best_eval  = evaluation
            print(f"  → New best: productivity={evaluation.productivity:.6f}, "
                  f"purity={evaluation.purity:.4f}, recovery={evaluation.recovery:.4f}")
    except RuntimeError as e:
        print(f"  → No feasible solution found: {e}")

if best_point is None:
    print("\nNo feasible solution found across all starts.")
else:
    print(f"\n{'='*40}")
    print(f"Best solution across {N_START} starts:")
    print(f"  Productivity = {best_eval.productivity:.6f} mol CO₂/(m³·s)")
    print(f"  Purity       = {best_eval.purity:.4f}")
    print(f"  Recovery     = {best_eval.recovery:.4f}")
    print(f"  Energy       = {best_eval.energy:.4f} kWh/tonne CO₂")

<![CDATA[## Results Analysis

### Key Findings

The optimizer converged to the initial point (best point from the data generation dataset). The trust-region radius collapsed from 0.125 to 0.001 over 20 iterations, with no neighboring point improving the objective while maintaining feasibility. This indicates the starting point is a **local optimum** within the sampled design space.

**Optimal operating conditions:**
| Variable | Value | Unit |
|----------|-------|------|
| P0 | 107,544 | Pa |
| ndot | 1.945 | mol/s |
| tads | 385.6 | s |
| alpha | 0.190 | — |
| beta | 0.353 | — |
| PI | 26,011 | Pa |
| Pl | 7,939 | Pa |

**Performance:**
| Metric | Value | Constraint |
|--------|-------|------------|
| Productivity | 8.3×10⁻⁵ mol CO₂/(m³·s) | — |
| Purity | 0.270 | ≥ 0.25 ✓ |
| Recovery | 1.005 | ≥ 0.50 ✓ |
| Energy | 254.4 kWh/tonne CO₂ | — |

### Note on Purity

Across ~800 Sobol-sampled points with material 16, the maximum observed purity was 0.270. This suggests that achieving 90% CO₂ purity with this adsorbent material under the current operating bounds is not feasible. Higher purity may require a different adsorbent material or expanded pressure/temperature ranges.]]>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../logs/optimization_iterations.csv")

# Extract the last optimization run (starts at the last iter=1)
starts = df.index[df["iter"] == 1].tolist()
df_last = df.iloc[starts[-1]].to_frame().T if len(starts) > 0 else df
df_last = df.iloc[starts[-1]:]

print(f"Plotting last run: {len(df_last)} iterations")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle("Trust-Region SLP Optimization Convergence", fontsize=14)

accepted = df_last["accepted"].astype(bool)

# 1. Objective
ax = axes[0, 0]
ax.plot(df_last["iter"], df_last["objective"], marker="o", color="steelblue")
ax.scatter(df_last["iter"][accepted], df_last["objective"][accepted],
           color="green", zorder=5, label="Accepted", s=60)
ax.scatter(df_last["iter"][~accepted], df_last["objective"][~accepted],
           color="red", marker="x", zorder=5, label="Rejected", s=60)
ax.set_xlabel("Iteration"); ax.set_ylabel("Objective"); ax.set_title("Objective")
ax.legend(fontsize=8); ax.grid(True)

# 2. Purity & Recovery
ax = axes[0, 1]
ax.plot(df_last["iter"], df_last["purity"],    marker="o", label="Purity",    color="tab:blue")
ax.plot(df_last["iter"], df_last["recovery"],  marker="s", label="Recovery",  color="tab:orange")
ax.axhline(purity_min,   linestyle="--", color="tab:blue",   alpha=0.5, label=f"Purity min={purity_min}")
ax.axhline(recovery_min, linestyle="--", color="tab:orange", alpha=0.5, label=f"Recovery min={recovery_min}")
ax.set_xlabel("Iteration"); ax.set_ylabel("Value"); ax.set_title("Purity & Recovery")
ax.legend(fontsize=8); ax.grid(True)

# 3. CSS Error
ax = axes[0, 2]
css_vals = df_last["css_error"]
if css_vals.max() > 0:
    ax.plot(df_last["iter"], css_vals, marker="o", color="purple")
    ax.set_yscale("log")
else:
    ax.plot(df_last["iter"], css_vals, marker="o", color="purple")
    ax.set_ylabel("CSS Error (all zero)")
ax.axhline(css_tol, linestyle="--", color="gray", label=f"CSS tol={css_tol}")
ax.set_xlabel("Iteration"); ax.set_title("CSS Convergence")
ax.legend(fontsize=8); ax.grid(True)

# 4. Trust-region radius
ax = axes[1, 0]
ax.plot(df_last["iter"], df_last["delta"], marker="o", color="darkorange")
ax.set_xlabel("Iteration"); ax.set_ylabel("Δ (trust radius)"); ax.set_title("Trust-Region Radius")
ax.set_yscale("log")
ax.grid(True)

# 5. Productivity
ax = axes[1, 1]
ax.plot(df_last["iter"], df_last["productivity"], marker="o", color="green")
ax.set_xlabel("Iteration"); ax.set_ylabel("Productivity [mol CO₂/(m³·s)]"); ax.set_title("Productivity")
ax.grid(True)

# 6. Energy
ax = axes[1, 2]
ax.plot(df_last["iter"], df_last["energy"], marker="o", color="crimson")
ax.set_xlabel("Iteration"); ax.set_ylabel("Energy [kWh/tonne CO₂]"); ax.set_title("Energy")
ax.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

units = {
    'L': 'm', 'P0': 'Pa', 'ndot': 'mol/s',
    'tads': 's', 'alpha': '—', 'beta': '—',
    'PI': 'Pa', 'Pl': 'Pa',
}

result = pd.DataFrame({
    "Variable":      VARIABLE_ORDER,
    "Optimal Value": [best_point[k] * SCALE[k] for k in VARIABLE_ORDER],
    "Unit":          [units[k] for k in VARIABLE_ORDER],
}).set_index("Variable")

result["Optimal Value"] = result["Optimal Value"].map(lambda x: f"{x:.4g}")
result

In [ ]:
perf = pd.DataFrame({
    "Metric": ["Productivity", "Purity", "Recovery", "Energy", "Objective"],
    "Value": [
        f"{best_eval.productivity:.4f} mol CO₂/(m³·s)",
        f"{best_eval.purity:.4f}",
        f"{best_eval.recovery:.4f}",
        f"{best_eval.energy:.4f} kWh/tonne CO₂",
        f"{objective_value(best_eval, energy_weight):.4f}",
    ],
    "Constraint": [
        "—",
        f"≥ {purity_min}  {'✓' if best_eval.purity >= purity_min else '✗'}",
        f"≥ {recovery_min}  {'✓' if best_eval.recovery >= recovery_min else '✗'}",
        "—",
        "—",
    ]
}).set_index("Metric")

perf

## Multi-Stage PSA for High Purity

Single-stage PSA with the available adsorbent materials achieves a maximum CO2 purity of ~0.53 (material 13) or ~0.27 (material 16). To reach >90% purity, we employ a **multi-stage PSA** approach where the heavy product (CO2-enriched stream) from one stage feeds into the next.

The feed composition parameter `y0` in `PSASimulator_local` has been made configurable to support this.

In [ ]:
import subprocess, json

def run_psa_stage(mat_index, N, vars_dict, y0=0.15):
    """Run a single PSA stage via Julia CLI and return results."""
    cmd = [
        "julia", "--project=..",
        "../scripts/evaluate_psa_point.jl",
        str(mat_index), str(N),
        *(str(vars_dict[k]) for k in VARIABLE_ORDER),
        "false", str(y0),
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True, check=True)
    productivity, energy, purity, recovery = [float(x) for x in proc.stdout.strip().split(",")]
    return {"productivity": productivity, "energy": energy, "purity": purity, "recovery": recovery}

# Operating conditions (material 13, best purity point)
mat_index_ms = 13
N_ms = 10
operating_point = {
    'L': 1.0, 'P0': 107544.0, 'ndot': 1.945, 'tads': 385.6,
    'alpha': 0.190, 'beta': 0.353, 'PI': 26011.0, 'Pl': 7939.0,
}

# Run multi-stage PSA
stages = []
y0_feed = 0.15  # flue gas CO2 fraction

for stage_num in range(1, 4):
    result = run_psa_stage(mat_index_ms, N_ms, operating_point, y0=y0_feed)
    result["stage"] = stage_num
    result["feed_y0"] = y0_feed
    stages.append(result)
    print(f"Stage {stage_num}: feed_y0={y0_feed:.4f} -> purity={result['purity']:.4f}, "
          f"recovery={result['recovery']:.4f}, productivity={result['productivity']:.6f}")
    y0_feed = result["purity"]  # feed next stage with this stage's heavy product

print(f"\nFinal purity after {len(stages)} stages: {stages[-1]['purity']:.4f}")

In [ ]:
import pandas as pd

stages_df = pd.DataFrame(stages)
stages_df = stages_df[["stage", "feed_y0", "purity", "recovery", "productivity", "energy"]]
stages_df.columns = ["Stage", "Feed CO2 Fraction", "Output Purity", "Recovery", "Productivity", "Energy (kWh/t)"]
stages_df = stages_df.set_index("Stage")
stages_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Multi-Stage PSA Performance (Material 13)", fontsize=14)

stage_nums = [s["stage"] for s in stages]

# Purity progression
ax = axes[0]
ax.bar(stage_nums, [s["purity"] for s in stages], color=["#4C72B0", "#55A868", "#C44E52"])
ax.axhline(0.90, linestyle="--", color="gray", label="90% target")
ax.set_xlabel("Stage"); ax.set_ylabel("CO2 Purity")
ax.set_title("Purity Progression"); ax.legend(); ax.set_xticks(stage_nums)

# Recovery
ax = axes[1]
ax.bar(stage_nums, [s["recovery"] for s in stages], color=["#4C72B0", "#55A868", "#C44E52"])
ax.set_xlabel("Stage"); ax.set_ylabel("Recovery")
ax.set_title("Recovery per Stage"); ax.set_xticks(stage_nums)

# Cumulative recovery
cumulative_recovery = []
cr = 1.0
for s in stages:
    cr *= s["recovery"]
    cumulative_recovery.append(cr)
ax = axes[2]
ax.bar(stage_nums, cumulative_recovery, color=["#4C72B0", "#55A868", "#C44E52"])
ax.set_xlabel("Stage"); ax.set_ylabel("Cumulative Recovery")
ax.set_title("Overall Recovery"); ax.set_xticks(stage_nums)

plt.tight_layout()
plt.show()

print(f"Overall recovery (3 stages): {cumulative_recovery[-1]:.4f}")
print(f"Final purity: {stages[-1]['purity']:.4f}")

## Multi-Stage PSA Optimization

We now **optimize** the 3-stage PSA configuration end-to-end. The 8 operating variables (L, P0, ndot, tads, α, β, PI, Pl) are **shared across all stages** (stage-shared design); only the feed composition `y0` chains from stage to stage (stage k's output purity → stage k+1's feed).

### Aggregated metrics (fed to the optimizer)

| Metric | Aggregation | Reason |
|---|---|---|
| **Productivity** | stage-3 only | Final product stream throughput |
| **Energy** | Σ per-stage | Each stage has its own compressor/blower duty |
| **Purity** | stage-3 (= final) | Purity is concentration — last stage overwrites |
| **Recovery** | Π per-stage | Fraction-capture compounds multiplicatively |

### Constraints

| Constraint | Value |
|---|---|
| Final purity (stage-3) | ≥ **0.90** |
| Cumulative recovery | ≥ **0.85** |
| Pressure ordering | Pl ≤ PI ≤ P0 |

### Materials

Optimization is run for material **8** (no prior data) and material **13** (baseline: notebook multi-stage simulation reached purity 0.955 / cumulative recovery 0.569).

### Method

Trust-region SLP with **20 random restarts** per material, bounds from Haghpanah et al. (2013) and related PSA literature.

⚠️ **Note**: cumulative recovery ≥ 0.85 requires per-stage recovery averaging ≥ 0.947, which is very tight. Some or all restarts may fail to find a feasible point.

In [ ]:
import os
from psa_pyomo.process.multi_stage_cycle_model import MultiStageCycleSimulator

# --- Multi-stage optimization parameters ---
materials_to_run = [8, 13]
num_stages       = 3
initial_y0       = 0.15     # flue gas CO2 fraction
N_START_MS       = 1        # 1 warm + 4 random
N_grid_ms        = 10

purity_min_ms   = 0.90
recovery_min_ms = 0.85
css_tol_ms      = 1e-4

max_iter_ms             = 10
delta0_ms               = 0.25
fd_rel_step_ms          = 1e-3
energy_weight_ms        = 0.0
infeasibility_penalty_ms = 1e6

# Physical bounds (same as single-stage optimization)
bounds_physical_ms = {
    'L':    (1.0,   1.0  ),
    'P0':   (1e5,   4e5  ),
    'ndot': (0.5,   2.0  ),
    'tads': (60.0,  400.0),
    'alpha':(0.10,  0.40 ),
    'beta': (0.10,  0.40 ),
    'PI':   (2e4,   2e5  ),
    'Pl':   (2e3,   1e4  ),
}
bounds_ms = {k: (lb / SCALE[k], ub / SCALE[k]) for k, (lb, ub) in bounds_physical_ms.items()}

# --- Warm-start seeds from the surrogate optimization ---
# Source: multi_material_surrogate_pyomo.ipynb (3-stage surrogate optima,
# per-material best point with purity >= 0.90, recovery >= 0.85).
SURROGATE_WARM_START = {
    8: {
        'L':     1.0,
        'P0':    231208.85,
        'ndot':  1.9711,
        'tads':  285.55,
        'alpha': 0.1057,
        'beta':  0.3933,
        'PI':    63199.67,
        'Pl':    9830.23,
    },
    13: {
        'L':     1.0,
        'P0':    190089.99,
        'ndot':  1.9711,
        'tads':  322.99,
        'alpha': 0.1057,
        'beta':  0.3933,
        'PI':    115627.23,
        'Pl':    9830.23,
    },
}

print(f"Materials:        {materials_to_run}")
print(f"Stages:           {num_stages}")
print(f"N_START:          {N_START_MS} (1 warm + {N_START_MS - 1} random)")
print(f"Purity min:       {purity_min_ms}")
print(f"Recovery min:     {recovery_min_ms} (cumulative)")
print(f"Initial feed y0:  {initial_y0}")
print(f"Warm-start seeds: {sorted(SURROGATE_WARM_START.keys())}")

In [ ]:
def run_multistage_optimization(mat_idx, n_start, log_path):
    """Run multi-start multi-stage PSA optimization for a given material.

    Start 0 uses the surrogate-optimization warm-start seed (if available for
    this material); all subsequent starts are random restarts within the
    scaled bounds, with pressure ordering enforced.

    Returns (best_point, best_aggregated_eval, best_stages, best_start_idx).
    """

    # Clean log file for this material
    if os.path.exists(log_path):
        os.remove(log_path)

    cycle_cfg = CycleConfig(
        material_index=mat_idx,
        n_grid=N_grid_ms,
        css_tol=css_tol_ms,
        feed_y0=initial_y0,
    )
    base_sim = CycleSimulator(Path('..'), cycle_cfg)
    ms_sim   = MultiStageCycleSimulator(base_sim, num_stages=num_stages, initial_y0=initial_y0)

    opt_cfg = OptimizationConfig(
        purity_min=purity_min_ms,
        recovery_min=recovery_min_ms,
        css_tol=css_tol_ms,
        energy_weight=energy_weight_ms,
        fd_rel_step=fd_rel_step_ms,
        delta0=delta0_ms,
        max_iter=max_iter_ms,
        solver_name=solver,
        iter_log_path=log_path,
        infeasibility_penalty=infeasibility_penalty_ms,
    )

    # Placeholder initial_point — overwritten at the start of every iteration below.
    decision_space_ms = ColumnDecisionSpace(
        bounds=bounds_ms,
        initial_point={k: (bounds_ms[k][0] + bounds_ms[k][1]) / 2 for k in VARIABLE_ORDER},
    )

    best_point       = None
    best_eval        = None
    best_stages      = None
    best_start_idx   = None

    has_warm = mat_idx in SURROGATE_WARM_START

    for i in range(n_start):
        # --- Select initial point for this restart ---
        if i == 0 and has_warm:
            # Warm start from surrogate optimum (scale to optimizer units)
            warm_physical = SURROGATE_WARM_START[mat_idx]
            decision_space_ms.initial_point = {
                k: warm_physical[k] / SCALE[k] for k in VARIABLE_ORDER
            }
            start_label = "warm (surrogate)"
        else:
            # Random restart within scaled bounds
            decision_space_ms.initial_point = {
                k: random.uniform(bounds_ms[k][0], bounds_ms[k][1])
                for k in VARIABLE_ORDER
            }
            start_label = "random"

        # Enforce pressure ordering on the initial point: Pl <= PI <= P0.
        # (The warm start already respects this, so the branches below only
        # trigger for random restarts, but leaving the check in is harmless.)
        if decision_space_ms.initial_point["PI"] > decision_space_ms.initial_point["P0"]:
            decision_space_ms.initial_point["PI"] = decision_space_ms.initial_point["P0"] * 0.5
        if decision_space_ms.initial_point["Pl"] > decision_space_ms.initial_point["PI"]:
            decision_space_ms.initial_point["Pl"] = decision_space_ms.initial_point["PI"] * 0.5

        print(f"\n--- Material {mat_idx} | Start {i+1}/{n_start} [{start_label}] ---")

        # --- Run trust-region SLP from this initial point ---
        try:
            point, evaluation = run_optimization(ms_sim, decision_space_ms, opt_cfg)

            # Feasibility check (with a small tolerance for numerical slack)
            feasible = (evaluation.purity   >= purity_min_ms   - 1e-6 and
                        evaluation.recovery >= recovery_min_ms - 1e-6)

            if feasible and (best_eval is None or evaluation.productivity > best_eval.productivity):
                # Re-evaluate to capture per-stage details for later plots/tables
                ms_result = ms_sim.evaluate_with_stages(
                    {k: point[k] * SCALE[k] for k in VARIABLE_ORDER}
                )
                best_point     = point
                best_eval      = evaluation
                best_stages    = ms_result.stages
                best_start_idx = i
                print(f"  → New best [{start_label}]: "
                      f"purity={evaluation.purity:.4f}, "
                      f"recovery={evaluation.recovery:.4f}, "
                      f"productivity={evaluation.productivity:.6g}")
            elif not feasible:
                print(f"  → Infeasible result: "
                      f"purity={evaluation.purity:.4f} (need ≥{purity_min_ms}), "
                      f"recovery={evaluation.recovery:.4f} (need ≥{recovery_min_ms})")
            else:
                print(f"  → Feasible but not an improvement: "
                      f"productivity={evaluation.productivity:.6g}")

        except RuntimeError as e:
            print(f"  → run_optimization raised: {str(e)[:80]}")

    return best_point, best_eval, best_stages, best_start_idx


# === Run for both materials ===
results_by_material = {}

for mat_idx in materials_to_run:
    print(f"\n{'='*60}")
    print(f"   MULTI-STAGE OPTIMIZATION — Material {mat_idx}")
    print(f"{'='*60}")

    log_path = f"../logs/multistage_iterations_mat{mat_idx}.csv"
    bp, be, bs, bidx = run_multistage_optimization(mat_idx, N_START_MS, log_path)

    results_by_material[mat_idx] = {
        "point":      bp,
        "eval":       be,
        "stages":     bs,
        "start_idx":  bidx,
        "log_path":   log_path,
    }

    if be is None:
        print(f"\n❌ Material {mat_idx}: NO feasible solution across {N_START_MS} starts.")
    else:
        start_type = "warm (surrogate)" if bidx == 0 and mat_idx in SURROGATE_WARM_START else "random"
        print(f"\n✅ Material {mat_idx} best (start #{bidx+1} [{start_type}]):")
        print(f"   Final purity         = {be.purity:.4f}")
        print(f"   Cumulative recovery  = {be.recovery:.4f}")
        print(f"   Stage-3 productivity = {be.productivity:.6g} mol CO₂/(m³·s)")
        print(f"   Total energy         = {be.energy:.4f} kWh/tonne CO₂")

### Results: Optimal operating conditions (per material)

In [ ]:
import pandas as pd

units_ms = {
    'L': 'm', 'P0': 'Pa', 'ndot': 'mol/s',
    'tads': 's', 'alpha': '—', 'beta': '—',
    'PI': 'Pa', 'Pl': 'Pa',
}

rows = []
for mat_idx, r in results_by_material.items():
    if r["point"] is None:
        rows.append({"Material": mat_idx, **{k: "—" for k in VARIABLE_ORDER}, "Status": "INFEASIBLE"})
    else:
        rec = {"Material": mat_idx, "Status": "feasible"}
        for k in VARIABLE_ORDER:
            val = r["point"][k] * SCALE[k]
            rec[k] = f"{val:.4g}"
        rows.append(rec)

opt_cond_df = pd.DataFrame(rows).set_index("Material")
opt_cond_df[["Status"] + VARIABLE_ORDER]

### Aggregate performance (per material)

In [ ]:
perf_rows = []
for mat_idx, r in results_by_material.items():
    e = r["eval"]
    if e is None:
        perf_rows.append({
            "Material": mat_idx, "Final Purity": "—", "Cumul. Recovery": "—",
            "Stage-3 Productivity": "—", "Total Energy (kWh/t)": "—",
            "Purity ≥ 0.90": "—", "Recovery ≥ 0.85": "—",
        })
    else:
        perf_rows.append({
            "Material": mat_idx,
            "Final Purity":          f"{e.purity:.4f}",
            "Cumul. Recovery":       f"{e.recovery:.4f}",
            "Stage-3 Productivity":  f"{e.productivity:.6g}",
            "Total Energy (kWh/t)":  f"{e.energy:.4f}",
            "Purity ≥ 0.90":         "✓" if e.purity   >= purity_min_ms   else "✗",
            "Recovery ≥ 0.85":       "✓" if e.recovery >= recovery_min_ms else "✗",
        })

perf_df = pd.DataFrame(perf_rows).set_index("Material")
perf_df

### Per-stage breakdown at the optimum

In [ ]:
stage_rows = []
for mat_idx, r in results_by_material.items():
    if r["stages"] is None:
        continue
    y0 = initial_y0
    for k, ev in enumerate(r["stages"], start=1):
        stage_rows.append({
            "Material": mat_idx,
            "Stage": k,
            "Feed y0":       f"{y0:.4f}",
            "Output Purity": f"{ev.purity:.4f}",
            "Recovery":      f"{ev.recovery:.4f}",
            "Productivity":  f"{ev.productivity:.6g}",
            "Energy":        f"{ev.energy:.4f}",
        })
        y0 = ev.purity

stage_df = pd.DataFrame(stage_rows).set_index(["Material", "Stage"])
stage_df

### Convergence plots (best start per material)

For each material, the 6-panel view shows the trajectory of the best restart out of all 20 runs. Aggregated purity = final (stage-3) purity; recovery = cumulative (Π per-stage); energy = total across 3 stages.

In [ ]:
def slice_best_run(log_path, best_idx):
    """Extract rows corresponding to the best-start run from the iteration log."""
    df = pd.read_csv(log_path)
    starts = df.index[df["iter"] == 1].tolist()
    if best_idx is None or best_idx >= len(starts):
        return None
    start_row = starts[best_idx]
    end_row   = starts[best_idx + 1] if (best_idx + 1) < len(starts) else len(df)
    return df.iloc[start_row:end_row].reset_index(drop=True)


for mat_idx, r in results_by_material.items():
    if r["eval"] is None:
        print(f"Material {mat_idx}: infeasible — skipping convergence plot.")
        continue

    df_best = slice_best_run(r["log_path"], r["start_idx"])
    if df_best is None or len(df_best) == 0:
        print(f"Material {mat_idx}: no log rows for best start — skipping.")
        continue

    accepted = df_best["accepted"].astype(bool)

    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    fig.suptitle(f"Multi-Stage Trust-Region SLP — Material {mat_idx} (best restart #{r['start_idx']+1})",
                 fontsize=14)

    # 1. Objective
    ax = axes[0, 0]
    ax.plot(df_best["iter"], df_best["objective"], marker="o", color="steelblue")
    ax.scatter(df_best["iter"][accepted], df_best["objective"][accepted],
               color="green", zorder=5, label="Accepted", s=60)
    ax.scatter(df_best["iter"][~accepted], df_best["objective"][~accepted],
               color="red", marker="x", zorder=5, label="Rejected", s=60)
    ax.set_xlabel("Iteration"); ax.set_ylabel("Objective (stage-3 productivity)")
    ax.set_title("Objective"); ax.legend(fontsize=8); ax.grid(True)

    # 2. Final Purity & Cumulative Recovery
    ax = axes[0, 1]
    ax.plot(df_best["iter"], df_best["purity"],   marker="o", label="Final Purity (stage-3)", color="tab:blue")
    ax.plot(df_best["iter"], df_best["recovery"], marker="s", label="Cumul. Recovery",        color="tab:orange")
    ax.axhline(purity_min_ms,   linestyle="--", color="tab:blue",   alpha=0.5, label=f"Purity min={purity_min_ms}")
    ax.axhline(recovery_min_ms, linestyle="--", color="tab:orange", alpha=0.5, label=f"Recovery min={recovery_min_ms}")
    ax.set_xlabel("Iteration"); ax.set_ylabel("Value")
    ax.set_title("Purity & Recovery"); ax.legend(fontsize=8); ax.grid(True)

    # 3. CSS Error (max across stages)
    ax = axes[0, 2]
    css_vals = df_best["css_error"]
    ax.plot(df_best["iter"], css_vals, marker="o", color="purple")
    if css_vals.max() > 0:
        ax.set_yscale("log")
    ax.axhline(css_tol_ms, linestyle="--", color="gray", label=f"CSS tol={css_tol_ms}")
    ax.set_xlabel("Iteration"); ax.set_ylabel("CSS Error (worst stage)")
    ax.set_title("CSS Convergence"); ax.legend(fontsize=8); ax.grid(True)

    # 4. Trust-region radius
    ax = axes[1, 0]
    ax.plot(df_best["iter"], df_best["delta"], marker="o", color="darkorange")
    ax.set_xlabel("Iteration"); ax.set_ylabel("Δ (trust radius)")
    ax.set_title("Trust-Region Radius"); ax.set_yscale("log"); ax.grid(True)

    # 5. Stage-3 Productivity
    ax = axes[1, 1]
    ax.plot(df_best["iter"], df_best["productivity"], marker="o", color="green")
    ax.set_xlabel("Iteration"); ax.set_ylabel("Stage-3 Productivity")
    ax.set_title("Productivity"); ax.grid(True)

    # 6. Total Energy (sum across stages)
    ax = axes[1, 2]
    ax.plot(df_best["iter"], df_best["energy"], marker="o", color="crimson")
    ax.set_xlabel("Iteration"); ax.set_ylabel("Total Energy [kWh/tonne CO₂]")
    ax.set_title("Energy"); ax.grid(True)

    plt.tight_layout()
    plt.show()

### Per-stage breakdown at the optimum (per material)

For each material's best operating point: how purity builds up across stages, recovery per stage, cumulative recovery progression, and energy per stage.

In [ ]:
import numpy as np

feasible_mats = [m for m, r in results_by_material.items() if r["stages"] is not None]

if len(feasible_mats) == 0:
    print("No feasible solutions to plot.")
else:
    for mat_idx in feasible_mats:
        r = results_by_material[mat_idx]
        stages = r["stages"]
        stage_nums = list(range(1, len(stages) + 1))

        purities  = [s.purity       for s in stages]
        recoveries = [s.recovery    for s in stages]
        prods     = [s.productivity for s in stages]
        energies  = [s.energy       for s in stages]

        cumul = []
        c = 1.0
        for rec in recoveries:
            c *= rec
            cumul.append(c)

        fig, axes = plt.subplots(1, 4, figsize=(18, 4))
        fig.suptitle(f"Per-Stage Breakdown — Material {mat_idx}", fontsize=14)

        colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B2", "#CCB974"][:len(stages)]

        # Purity progression
        ax = axes[0]
        feed_seq = [initial_y0] + purities[:-1]
        ax.bar(stage_nums, purities, color=colors, label="Output purity")
        ax.plot(stage_nums, feed_seq, "o--", color="black", label="Feed y0", alpha=0.6)
        ax.axhline(purity_min_ms, linestyle="--", color="red", alpha=0.6, label=f"Target {purity_min_ms}")
        ax.set_xlabel("Stage"); ax.set_ylabel("CO₂ Fraction")
        ax.set_title("Purity Progression"); ax.set_xticks(stage_nums)
        ax.legend(fontsize=8); ax.set_ylim(0, 1.05)

        # Recovery per stage
        ax = axes[1]
        ax.bar(stage_nums, recoveries, color=colors)
        ax.set_xlabel("Stage"); ax.set_ylabel("Recovery")
        ax.set_title("Recovery per Stage"); ax.set_xticks(stage_nums)
        ax.set_ylim(0, 1.05)

        # Cumulative recovery
        ax = axes[2]
        ax.bar(stage_nums, cumul, color=colors)
        ax.axhline(recovery_min_ms, linestyle="--", color="red", alpha=0.6, label=f"Target {recovery_min_ms}")
        ax.set_xlabel("Stage"); ax.set_ylabel("Cumulative Recovery")
        ax.set_title("Cumulative Recovery"); ax.set_xticks(stage_nums)
        ax.legend(fontsize=8); ax.set_ylim(0, 1.05)

        # Energy per stage
        ax = axes[3]
        ax.bar(stage_nums, energies, color=colors)
        ax.set_xlabel("Stage"); ax.set_ylabel("Energy [kWh/tonne CO₂]")
        ax.set_title("Energy per Stage"); ax.set_xticks(stage_nums)

        plt.tight_layout()
        plt.show()

        print(f"Material {mat_idx}: final purity={purities[-1]:.4f}, "
              f"cumul. recovery={cumul[-1]:.4f}, total energy={sum(energies):.2f} kWh/t")

### Material comparison

Side-by-side comparison of the two materials across the four KPIs. This view surfaces which adsorbent (at its own optimum) delivers better overall PSA performance.

In [ ]:
if len(feasible_mats) < 1:
    print("No feasible solutions — skipping material comparison plot.")
else:
    mats = feasible_mats
    purities   = [results_by_material[m]["eval"].purity       for m in mats]
    recoveries = [results_by_material[m]["eval"].recovery     for m in mats]
    prods      = [results_by_material[m]["eval"].productivity for m in mats]
    energies   = [results_by_material[m]["eval"].energy       for m in mats]

    labels = [f"Mat {m}" for m in mats]
    x = np.arange(len(mats))
    bar_colors = ["#4C72B0", "#C44E52", "#55A868", "#8172B2"][:len(mats)]

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle("Material Comparison — Multi-Stage PSA Optima", fontsize=14)

    ax = axes[0]
    ax.bar(x, purities, color=bar_colors)
    ax.axhline(purity_min_ms, linestyle="--", color="red", alpha=0.6, label=f"Target {purity_min_ms}")
    ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_ylim(0, 1.05); ax.set_ylabel("Final Purity")
    ax.set_title("Final (stage-3) Purity"); ax.legend(fontsize=8)
    for xi, v in zip(x, purities):
        ax.text(xi, v + 0.02, f"{v:.3f}", ha="center", fontsize=9)

    ax = axes[1]
    ax.bar(x, recoveries, color=bar_colors)
    ax.axhline(recovery_min_ms, linestyle="--", color="red", alpha=0.6, label=f"Target {recovery_min_ms}")
    ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_ylim(0, 1.05); ax.set_ylabel("Cumulative Recovery")
    ax.set_title("Cumulative Recovery"); ax.legend(fontsize=8)
    for xi, v in zip(x, recoveries):
        ax.text(xi, v + 0.02, f"{v:.3f}", ha="center", fontsize=9)

    ax = axes[2]
    ax.bar(x, prods, color=bar_colors)
    ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_ylabel("Stage-3 Productivity [mol CO₂/(m³·s)]")
    ax.set_title("Productivity")
    for xi, v in zip(x, prods):
        ax.text(xi, v * 1.02, f"{v:.2e}", ha="center", fontsize=9)

    ax = axes[3]
    ax.bar(x, energies, color=bar_colors)
    ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_ylabel("Total Energy [kWh/tonne CO₂]")
    ax.set_title("Total Energy (Σ stages)")
    for xi, v in zip(x, energies):
        ax.text(xi, v * 1.02, f"{v:.1f}", ha="center", fontsize=9)

    plt.tight_layout()
    plt.show()